# Initialization

In [1]:
# Laborers
import os
import pandas as pd
import numpy as np

# Image workers
import cv2 as cv
from PIL import Image

# CNN
## CNN infrastructure
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

## CNN image workers
from torchvision import transforms, datasets
from torchvision.transforms import v2

device = torch.device(torch.accelerator.current_accelerator().type)

## Preprocessing

In [2]:
crop_coords = [
    (85, 340, 234, 560),
    (85, 610, 234, 830),
    (85, 870, 234, 1090),
    (85, 1140, 234, 1360),
    (85, 1400, 234, 1620),
    (85, 1670, 234, 1890)
]

# Set transformer.
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Attach label (key) to index (value).
id = {label: index for label, index in 
      enumerate(sorted(os.listdir("training/")))}

# Run CNN.

In [3]:
# Initialize basic neural network structure.
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(46656, 10) 

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x
    
# Activate.
model = Net().to(device)
model.load_state_dict(torch.load("gcq.pth", weights_only=True))

<All keys matched successfully>

In [4]:
results = []

for result in os.listdir(r"cq_results"):
    # Summon and crop.
    img = Image.open(f"cq_results/{result}")

    for coords in crop_coords:
        # Cropping.
        crop = img.crop(coords)
        crop = transform(crop).unsqueeze(0) 

        # Prediction and storing.
        output = model(crop.to(device))
        _, predicted = torch.max(output, 1)
        results.append(predicted[0].item())

In [ ]:
# Render into dataframe.
results_id = [id.get(item, item) for item in results]
results_df = pd.DataFrame({"drop" : results_id,
              "level" : np.tile([6,5,4,3,2,1], int(len(results)/6))})

results_df.to_csv("drops_csv/gcq_final.csv")